In [1]:
import pandas as pd
import os

# 0
# remove any previously generated csvs  
generated_files = ['data_cleaned.csv', 'data_features.csv']
for f in generated_files:
    if os.path.exists(f):
        os.remove(f)
        print(f"Deleted {f}")
    else:
        print(f"{f} not found, skipping")
print()

Deleted data_cleaned.csv
Deleted data_features.csv



### DATA CLEANING

In [2]:
#  1 
# csv load and size test
df = pd.read_csv('zikmund-fisher-replicated-study_all_tidy(12).csv')
initial_participants = df['participantId'].nunique()
initial_rows = len(df)
print(f"Loaded {initial_rows} rows, {initial_participants} participants")

Loaded 8429 rows, 129 participants


In [3]:
# drop all rows for Zoe
manual_exclude = ['69d84cf08773d16d3d083f65']
df = df[~df['participantId'].isin(manual_exclude)]
excluded_count = initial_participants - df['participantId'].nunique()
print(f"Manually excluded {excluded_count} participant(s)")

Manually excluded 0 participant(s)


In [5]:
#  2 
# find the age response for each participant 
# remove all rows for  participant age is less than 18 
age_rows = df[df['responsePrompt'] == 'Please enter your age:'].copy()
age_rows['age'] = pd.to_numeric(age_rows['answer'], errors='coerce')
underage_ids = age_rows.loc[age_rows['age'] < 18, 'participantId'].unique()
df = df[~df['participantId'].isin(underage_ids)]
print(f"Removed {len(underage_ids)} participant(s) for being underage")

Removed 0 participant(s) for being underage


In [6]:
# 3 
# if Prolific ID is blank/ nan 
# remove all rows for those ppl
prolific_rows = df[df['responsePrompt'] == 'Please enter your Prolific ID'].copy()
prolific_rows['prolific_id'] = prolific_rows['answer'].astype(str).str.strip()

# participants who answered the Prolific prompt w blank / nan
invalid_prolific_ids = prolific_rows.loc[
    (prolific_rows['prolific_id'] == '') |
    (prolific_rows['prolific_id'].str.lower() == 'nan'),
    'participantId'
].unique()

# participants who have no Prolific prompt row at all
all_ids = df['participantId'].unique()
ids_with_prolific_row = prolific_rows['participantId'].unique()
missing_prolific_ids = set(all_ids) - set(ids_with_prolific_row)

no_prolific_ids = set(invalid_prolific_ids) | missing_prolific_ids
df = df[~df['participantId'].isin(no_prolific_ids)]
print(f"Removed {len(no_prolific_ids)} participant(s) for missing/invalid Prolific ID")

Removed 7 participant(s) for missing/invalid Prolific ID


In [7]:
#  4
# check that all ppl has at least one row for each critical trial
# category 
# if any category is entirely missing for a person,,,,, remove them
critical_categories = {
    'Platelet':                   lambda tid: 'Platelet' in tid,
    'ALT':                        lambda tid: 'ALT' in tid,
    'Creatinine':                 lambda tid: 'Creatinine' in tid,
    'subjective-numeracy-scale':  lambda tid: tid == 'subjective-numeracy-scale',
    'chews-screening-question':   lambda tid: tid == 'chews-screening-question',
    'familiarity-question':       lambda tid: tid == 'familiarity-question',
    '$graph-literacy-scale':      lambda tid: tid.startswith('$graph-literacy-scale'),
}

remaining_ids = df['participantId'].unique()
ids_missing_critical = set()

for pid in remaining_ids:
    pid_trials = df.loc[df['participantId'] == pid, 'trialId'].unique()
    for cat_name, cat_match in critical_categories.items():
        if not any(cat_match(t) for t in pid_trials):
            print(f"  WARNING: participant {pid} missing data for '{cat_name}'")
            ids_missing_critical.add(pid)
            break  # one missing category is enough to flag

df = df[~df['participantId'].isin(ids_missing_critical)]
print(f"\nRemoved {len(ids_missing_critical)} participant(s) for missing critical data")


Removed 0 participant(s) for missing critical data


In [8]:
# 5 
# return how many ppl and rows remain after all cleaning steps
final_participants = df['participantId'].nunique()
final_rows = len(df)
print(f"{'='*50}")
print(f"Final dataset: {final_rows} rows, {final_participants} participants")
print(f"{'='*50}")

Final dataset: 8354 rows, 122 participants


In [9]:
# 6 
# write and save the cleaned DF to a new CSV  
df.to_csv('data_cleaned.csv', index=False)
print("Saved cleaned data to data_cleaned.csv")

Saved cleaned data to data_cleaned.csv


In [10]:
import numpy as np

# load cleaned data 
df = pd.read_csv('data_cleaned.csv')
participants = df['participantId'].unique()
result = pd.DataFrame({'participantId': participants})

# define test-type keyword mapping for prompt matching
# map each medical test type to the keyword used in responsePrompt text
test_type_keywords = {
    'Platelet':   'platelet',
    'ALT':        'ALT',
    'Creatinine': 'creatinine',
}

# identify medical trial rows
medical_trials = [t for t in df['trialId'].unique()
                  if any(t.startswith(tt + '_') for tt in test_type_keywords)]
print(f"Loaded {len(participants)} participants, {len(medical_trials)} medical trials")

Loaded 122 participants, 24 medical trials


### Feature engineering/Data Manipulation 

In [11]:
# 1  
# for each person & medical trial,,,, finds the "alarming" and "urgent"
# response rows n converts both to numbers n then averages them to get a single
# perceived_urgency score

# creates one column per trial  
for trial in sorted(medical_trials):
    test_type = trial.split('_')[0]
    kw = test_type_keywords[test_type]

    trial_df = df[df['trialId'] == trial]

    alarming = trial_df[trial_df['responsePrompt'].str.contains('alarming', case=False, na=False)]
    urgent   = trial_df[trial_df['responsePrompt'].str.contains('urgent', case=False, na=False)]

    alarming_vals = alarming.set_index('participantId')['answer'].astype(float)
    urgent_vals   = urgent.set_index('participantId')['answer'].astype(float)

    combined = pd.DataFrame({'alarming': alarming_vals, 'urgent': urgent_vals})
    combined[f'{trial}_urgency'] = combined.mean(axis=1)

    result = result.merge(
        combined[[f'{trial}_urgency']].reset_index(),
        on='participantId', how='left'
    )

print(f"STEP 1 done: {len([c for c in result.columns if c.endswith('_urgency')])} urgency columns")

STEP 1 done: 24 urgency columns


In [12]:
# 2  
# for each test type and display format n  computes how much more urgent the
# "further from normal" value felt compared to the "slightly abnormal" value 
# this shows sensitivity to result severity within the same display format
test_types = sorted(test_type_keywords.keys())
display_formats = sorted({t.split('_')[2] for t in medical_trials})

for tt in test_types:
    for fmt in display_formats:
        further_col  = f'{tt}_further_{fmt}_urgency'
        slightly_col = f'{tt}_slightly_{fmt}_urgency'
        diff_col     = f'{tt}_{fmt}_urgency_diff'
        if further_col in result.columns and slightly_col in result.columns:
            result[diff_col] = result[further_col] - result[slightly_col]

diff_cols = [c for c in result.columns if c.endswith('_urgency_diff')]
print(f"STEP 2 done: {len(diff_cols)} urgency_diff columns")

STEP 2 done: 12 urgency_diff columns


In [13]:
# 3  
# for each participant & trial, find the "what you would do in response"
# answer and recode it into two categories: 
# "Willingness to wait" (doing nothing or waiting for a regular appointment) vs. 
# "Some form of immediate action" (seeking an urgent or emergency appointment).
willingness_to_wait = {
    "Nothing",
    "Talk to your doctor about this test result at your next regular appointment",
}
immediate_action = {
    "Ask to see your doctor at the first available appointment",
    "Go to a hospital or your doctor's office tomorrow",
    "Go to a hospital as soon as you can get free later today",
    "Go to a hospital immediately",
    "Go to a hospital immediately.",
}

def recode_intention(ans):
    ans_str = str(ans).strip()
    if ans_str in willingness_to_wait:
        return "Willingness to wait"
    elif ans_str in immediate_action:
        return "Some form of immediate action"
    return np.nan

for trial in sorted(medical_trials):
    test_type = trial.split('_')[0]
    trial_df = df[df['trialId'] == trial]
    intention_rows = trial_df[trial_df['responsePrompt'].str.contains('what you would do', case=False, na=False)]
    col_name = f'{trial}_intention'
    intention_map = intention_rows.set_index('participantId')['answer'].apply(recode_intention)
    intention_map.name = col_name
    result = result.merge(intention_map.reset_index(), on='participantId', how='left')

print(f"STEP 3 done: {len([c for c in result.columns if c.endswith('_intention')])} intention columns")

STEP 3 done: 24 intention columns


In [14]:
#  4 
# score each participant on the 7-item graph literacy scale by comparing their
# answer to the correctAnswer for each question 
# the score is the count of correct responses (integer 0–7).
gl = df[df['trialId'].str.startswith('$graph-literacy-scale')].copy()
gl['correct'] = gl['answer'].astype(str).str.strip() == gl['correctAnswer'].astype(str).str.strip()
gl_scores = gl.groupby('participantId')['correct'].sum().astype(int).rename('graphical_literacy_score')
result = result.merge(gl_scores.reset_index(), on='participantId', how='left')
print(f"STEP 4 done: graphical_literacy_score (range {result['graphical_literacy_score'].min()}-{result['graphical_literacy_score'].max()})")

STEP 4 done: graphical_literacy_score (range 1-7)


In [15]:
#  5 
# extract the single-item CHEWS health literacy screening score for each person. 
# the answer is a numeric scale value
chews = df[df['trialId'] == 'chews-screening-question'].copy()
chews['answer_num'] = pd.to_numeric(chews['answer'], errors='coerce')
chews_scores = chews.groupby('participantId')['answer_num'].first().rename('health_literacy')
result = result.merge(chews_scores.reset_index(), on='participantId', how='left')
print("STEP 5 done: health_literacy")

STEP 5 done: health_literacy


In [16]:
# 6 
# average all SNS item responses for each participant into a single 
# composite score reflecting self-reported comfort with numbers and numerical info 
sns_df = df[df['trialId'] == 'subjective-numeracy-scale'].copy()
sns_df['answer_num'] = pd.to_numeric(sns_df['answer'], errors='coerce')
sns_scores = sns_df.groupby('participantId')['answer_num'].mean().rename('subjective_numeracy')
result = result.merge(sns_scores.reset_index(), on='participantId', how='left')
print("STEP 6 done: subjective_numeracy")

STEP 6 done: subjective_numeracy


In [17]:
# 7 
# extract each participant's self-reported familiarity with medical test
# results like the ones discussed in the survey (single numeric scale item).
fam = df[df['trialId'] == 'familiarity-question'].copy()
fam['answer_num'] = pd.to_numeric(fam['answer'], errors='coerce')
fam_scores = fam.groupby('participantId')['answer_num'].first().rename('familiarity')
result = result.merge(fam_scores.reset_index(), on='participantId', how='left')
print("STEP 7 done: familiarity")

STEP 7 done: familiarity


In [18]:
# 8 
# pull age, gender, ethnicity, race, and education from the demographic questions
# rows by matching keywords in responsePrompt 
# age is converted to number
demo = df[df['trialId'] == 'demographic-questions'].copy()
demo_map = {
    'age':       'age',
    'gender':    'gender',
    'ethnicity': 'ethnicity',
    'race':      'race',
    'education': 'education',
}
for col_name, keyword in demo_map.items():
    rows = demo[demo['responsePrompt'].str.contains(keyword, case=False, na=False)]
    vals = rows.drop_duplicates('participantId').set_index('participantId')['answer'].rename(col_name)
    if col_name == 'age':
        vals = pd.to_numeric(vals, errors='coerce')
    result = result.merge(vals.reset_index(), on='participantId', how='left')
print("STEP 8 done: demographics (age, gender, ethnicity, race, education)")

STEP 8 done: demographics (age, gender, ethnicity, race, education)


In [19]:
# 9 
# get each person's Prolific ID for linking back to the Prolific dashboard 

prolific = df[df['responsePrompt'] == 'Please enter your Prolific ID']
prolific_ids = prolific.drop_duplicates('participantId').set_index('participantId')['answer'].rename('prolific_id')
result = result.merge(prolific_ids.reset_index(), on='participantId', how='left')
print("STEP 9 done: prolific_id")

STEP 9 done: prolific_id


In [20]:
# 10
# save to CSV and print tt
# ══════════════════════════════════════════════════════════════════════════════
result.to_csv('data_features.csv', index=False)
print(f"{'='*60}")
print(f"Saved data_features.csv: {result.shape[0]} rows × {result.shape[1]} columns")
print(f"{'='*60}")
print("\nColumns:")
for c in result.columns:
    print(f"  {c}")

Saved data_features.csv: 122 rows × 71 columns

Columns:
  participantId
  ALT_further_block_urgency
  ALT_further_gradient_urgency
  ALT_further_simple_urgency
  ALT_further_table_urgency
  ALT_slightly_block_urgency
  ALT_slightly_gradient_urgency
  ALT_slightly_simple_urgency
  ALT_slightly_table_urgency
  Creatinine_further_block_urgency
  Creatinine_further_gradient_urgency
  Creatinine_further_simple_urgency
  Creatinine_further_table_urgency
  Creatinine_slightly_block_urgency
  Creatinine_slightly_gradient_urgency
  Creatinine_slightly_simple_urgency
  Creatinine_slightly_table_urgency
  Platelet_further_block_urgency
  Platelet_further_gradient_urgency
  Platelet_further_simple_urgency
  Platelet_further_table_urgency
  Platelet_slightly_block_urgency
  Platelet_slightly_gradient_urgency
  Platelet_slightly_simple_urgency
  Platelet_slightly_table_urgency
  ALT_block_urgency_diff
  ALT_gradient_urgency_diff
  ALT_simple_urgency_diff
  ALT_table_urgency_diff
  Creatinine_block_